<h3>구조 청킹과 시맨틱 청킹</h3>
<p><strong>개념</strong> 구조 청킹은 제목과 빈 줄 같은 문서 표식을 사용합니다. 시맨틱 청킹은 인접 문장의 임베딩 유사도가 낮아지는 위치를 경계 후보로 사용합니다.</p>

In [1]:
# %pip은 현재 커널에 환경 변수, LangChain 임베딩, 토큰 계산 패키지를 설치합니다.
# 버전을 고정해 OpenAIEmbeddings의 인자와 반환 형식을 동일하게 맞춥니다.
%pip install -q "python-dotenv==1.2.3" "langchain-openai==1.6.0" "tiktoken==0.14.0"


Note: you may need to restart the kernel to use updated packages.


In [2]:
# .env의 API 키를 읽고 256차원 임베딩 객체를 준비합니다. 키 값 자체는 출력하지 않습니다.
# 정규 표현식은 문장부호 뒤 공백이나 새 제목 앞을 경계로 원문을 문장 리스트로 나눕니다.
# (?<=[.!?])는 바로 앞의 문장부호를 확인하는 후방 탐색이고, \s+는 뒤따르는 공백을 뜻합니다.
# (?=# )는 '# '가 이어지는 위치를 확인하는 전방 탐색이며 제목 문자는 소비하지 않습니다.
from dotenv import find_dotenv, load_dotenv
from langchain_openai import OpenAIEmbeddings
from math import sqrt
import re, tiktoken

# 현재 폴더부터 상위 폴더까지 .env를 찾고, 읽은 키를 환경 변수로 등록합니다.
load_dotenv(find_dotenv(usecwd=True))
# dimensions=256은 문장 하나를 표현할 숫자 개수를 정합니다. 아직 API를 호출하지는 않습니다.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=256)
text = """# 배송
결제 다음 날 출고합니다. 제주 지역은 이틀이 더 걸립니다. # 환불
수령 후 7일 이내 신청합니다."""
# strip()으로 양끝 공백을 지우고, if s.strip()으로 빈 조각을 제외합니다.
sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+|(?=# )", text) if s.strip()]


<h3>실제 임베딩으로 경계 찾기</h3>
<p><strong>확인할 점</strong> LangChain의 <code>OpenAIEmbeddings.embed_documents()</code>로 <code>text-embedding-3-small</code>을 호출합니다. 로컬 토큰 수는 비용 규모를 확인하는 추정값입니다.</p>

In [3]:
# embed_documents()는 각 문장을 같은 차원의 실수 벡터로 반환합니다.
# 인접 벡터의 코사인 유사도가 임계값보다 낮은 위치를 경계로 삼아 청크를 만듭니다.
# local_tokens는 cl100k_base로 센 로컬 추정값이며 API 공급자가 반환한 사용량이 아닙니다.
vectors = embeddings.embed_documents(sentences)
encoding = tiktoken.get_encoding("cl100k_base")
local_tokens = sum(len(encoding.encode(sentence)) for sentence in sentences)
# 두 벡터의 같은 위치 숫자끼리 곱한 합을 각 벡터의 길이로 나누어 방향의 유사도를 구합니다.
def cosine(a: list[float], b: list[float]) -> float:
    return sum(x*y for x, y in zip(a, b)) / (sqrt(sum(x*x for x in a))*sqrt(sum(y*y for y in b)))
# zip(vectors, vectors[1:])은 현재 벡터와 바로 다음 벡터를 인접 쌍으로 묶습니다.
scores = [round(cosine(a, b), 3) for a, b in zip(vectors, vectors[1:])]
# 0.25는 이 예제의 관찰용 기준입니다. 모든 문서에 적합한 고정 정답은 아닙니다.
boundary_threshold = 0.25
chunks, start = [], 0
# 점수가 임계값보다 낮으면 의미가 달라졌다고 보고 i+1을 다음 청크의 시작 위치로 기록합니다.
# sentences[start:boundary]에서 boundary는 포함되지 않는 슬라이스 끝이므로 경계 문장은 다음 청크에 남습니다.
for boundary in [i + 1 for i, score in enumerate(scores) if score < boundary_threshold]:
    chunks.append(" ".join(sentences[start:boundary])); start = boundary
chunks.append(" ".join(sentences[start:]))
print("sentences:", sentences)
print("adjacent cosine:", scores, "chunks:", chunks, "local_token_estimate:", local_tokens)


sentences: ['# 배송\n결제 다음 날 출고합니다.', '제주 지역은 이틀이 더 걸립니다.', '# 환불\n수령 후 7일 이내 신청합니다.']
adjacent cosine: [0.244, 0.269] chunks: ['# 배송\n결제 다음 날 출고합니다.', '제주 지역은 이틀이 더 걸립니다. # 환불\n수령 후 7일 이내 신청합니다.'] local_token_estimate: 57


<h3>일부러 실패해 보기</h3>
<p><strong>실패 사례</strong> 임계값을 지나치게 높이면 거의 모든 문장 사이가 경계가 됩니다. 다음 셀은 이 과분할을 포착해 보여 줍니다.</p>

In [4]:
# 지나치게 높은 임계값을 넣어 거의 모든 문장이 분리되는 과분할을 재현합니다.
# enumerate()의 위치에 1을 더해 다음 문장이 시작되는 경계 번호를 만듭니다.
try:
    threshold = 0.999
    boundaries = [i + 1 for i, score in enumerate(scores) if score < threshold]
    assert len(boundaries) < len(sentences) - 1, "거의 모든 문장이 분리되었습니다"
except AssertionError as error:
    print(f"caught: {error}; boundaries={boundaries}")


caught: 거의 모든 문장이 분리되었습니다; boundaries=[1, 2]


<p><strong>수정 방법</strong> 같은 유사도에 관찰 가능한 기준값을 적용하고, 모든 문장이 분리되지 않았는지 검증합니다.</p>

In [5]:
# 관찰한 유사도에 맞춘 임계값으로 경계를 다시 계산하고 과분할 여부를 검증합니다.
# *boundaries는 경계 목록을 펼치며 zip()은 연속한 시작점과 끝점을 짝지어 반환합니다.
threshold = 0.25
boundaries = [i + 1 for i, score in enumerate(scores) if score < threshold]
assert len(boundaries) < len(sentences) - 1
points = [0, *boundaries, len(sentences)]
fixed_chunks = [" ".join(sentences[a:b]) for a, b in zip(points, points[1:])]
print({"threshold": threshold, "boundaries": boundaries, "chunks": fixed_chunks})


{'threshold': 0.25, 'boundaries': [1], 'chunks': ['# 배송\n결제 다음 날 출고합니다.', '제주 지역은 이틀이 더 걸립니다. # 환불\n수령 후 7일 이내 신청합니다.']}
